# Import

In [19]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, global_max_pool, global_add_pool
from utils import *
from sklearn.model_selection import train_test_split
import glob
import pandas as pd
import json
import os
import pickle
from torch_geometric.data import Data, Batch
import networkx as nx
import matplotlib.pyplot as plt
import math
import torch_scatter
from train_logic import test_epoch
import io
from torch_geometric.utils import to_networkx, subgraph
from torch_geometric.data import Batch
from tell import step
import copy

# Functions

In [20]:
def get_best_baseline_path(dataset_name):
    l = glob.glob(f'results/{dataset_name}/*/results.json')
    fl = [json.load(open(f)) for f in l]
    df = pd.DataFrame(fl)
    if df.shape[0] == 0: return None
    df['fname'] = l
    df = df.sort_values(by=['val_acc_mean', 'val_acc_std', 'test_acc_std'], ascending=[True,False,False])
    df = df[df.fname.str.contains('nogumbel=False')]
    fname = df.iloc[-1]['fname']
    fname = fname.replace('/results.json', '')
    return fname

def get_best_path(dataset_name):
    l = glob.glob(f'results_logic/{dataset_name}/*/*/results.json')
    fl = [json.load(open(f)) for f in l]
    df = pd.DataFrame(fl)
    if df.shape[0] == 0: return None
    df['fname'] = l
    df = df.sort_values(by=['val_acc_mean', 'val_acc_std', 'test_acc_std'], ascending=[True,False,False])
    df = df[df.fname.str.contains('nogumbel=False')]
    print(df.tail())
    fname = df.iloc[-1]['fname']
    fname = fname.replace('/results.json', '')
    return fname

def inverse_sigmoid(x):
    """Computes the inverse of the sigmoid function (logit function)."""
    return torch.log(x / (1 - x))

torch.no_grad()
def find_logic_rules(w, t_in, t_out, activations=None, max_rule_len=10, max_rules=100, min_support=5):
    w = w.clone()
    t_in = t_in.clone() if t_in is not None else None
    t_out = t_out.clone()
    t_out = t_out.item()
    ordering_scores = w

    # print("Activations:", activations)
    # print("Activations shape:", activations.shape if activations is not None else None)

    sorted_idxs = torch.argsort(ordering_scores, 0, descending=True) #Riordini i pesi in ordine decrescente
    #print("Sorted idxs:", sorted_idxs)

    mask = w > 1e-4 #Considero solo i pesi maggiori di una soglia
    # print("Mask shape:", mask.shape)

    if activations is not None:
        # print("Activations sum:", activations.sum(0))
        # print("Activations sum shape:", activations.sum(0).shape if activations is not None else None)
        mask = mask & (activations.sum(0) >= min_support) # Somma per colonna delle attivazioni per ogni feature. 
                                                          # Credo ignori i pesi che intervengono poco nelle attivazioni del BATCH che stiamo considerando
        # print("Updated Mask:", mask)

    total_result = set()

    # Filter and sort indices based on the mask
    idxs_to_visit = sorted_idxs[mask[sorted_idxs]]
    #print("Idxs to visit:", idxs_to_visit)

    if idxs_to_visit.numel() == 0:
        return total_result

    # Sort weights based on the filtered indices
    sorted_weights = w[idxs_to_visit]
    current_combination = []
    result = set()

    def find_logic_rules_recursive(index, current_sum):
        # Stop if the maximum number of rules has been reached
        if len(result) >= max_rules:
            return

        if len(current_combination) > max_rule_len:
            return

        # Check if the current combination satisfies the condition
        if current_sum >= t_out:
            c = idxs_to_visit[current_combination].cpu().detach().tolist()
            c = tuple(sorted(c))
            result.add(c)
            return

        # Prune if remaining weights can't satisfy t_out
        remaining_max_sum = current_sum + sorted_weights[index:].sum()
        if remaining_max_sum < t_out:
            return

        # Explore further combinations
        for i in range(index, idxs_to_visit.shape[0]):
            # Prune based on activations if provided
            if activations is not None and len(current_combination) > 0 and activations[:, idxs_to_visit[current_combination + [i]]].all(-1).sum().item() < min_support:
                #Capire cosa e' activations[:, idxs_to_visit[current_combination + [i]]].all(-1).sum().item()
                # Credo ignori i pesi che intervengono poco nelle attivazioni del BATCH che stiamo considerando
                continue

            current_combination.append(i)
            find_logic_rules_recursive(i + 1, current_sum + sorted_weights[i])
            current_combination.pop()

    # Start the recursive process
    find_logic_rules_recursive(0, 0)
    return result


def extract_rules(self, feature=None, activations=None, max_rule_len=float('inf'), max_rules=5, min_support=0, out_threshold=0.5):
    ws = self.weight
    #print("ws shape:", ws.shape)
    t_in = self.phi_in.t if self.phi_in is not None else None
    t_out = -self.b + inverse_sigmoid(torch.tensor(out_threshold)) 

    rules = []
    #Feature e' il range di feature output
    if feature is None:
        features = range(self.out_features)
    else:
        features = [feature]

    for i in features:
        w = ws[i].to('cpu')
        #print("w i w.shape:", w.shape)
        ti = t_in.to('cpu') if t_in is not None else None
        to = t_out[i].to('cpu')
        rules.append(find_logic_rules(w, ti, to, activations, max_rule_len, max_rules, min_support))

    return rules

def plot_activations(batch_ids, batch, attr, save_path=None):
    if type(batch_ids) != list:
        batch_ids = [batch_ids]
    num_ids = len(batch_ids)
    cols = 5
    rows = math.ceil(num_ids / cols)
    
    fig, axs = plt.subplots(rows, cols, figsize=(16*5, 8 * rows))
    
    if type(axs) != np.ndarray: axs = np.array([axs])
    # Flatten axs if it's 2D to simplify indexing
    axs = axs.flatten()
    for ax in axs:
        ax.set_axis_off()
    for i, batch_id in enumerate(batch_ids):
        node_mask = batch.batch == batch_id  # Get nodes where batch == 0
        if node_mask.float().sum() == 0: continue
        node_indices = torch.nonzero(node_mask, as_tuple=True)[0]
        
        subgraph_edge_mask = (batch.batch[batch.edge_index[0]] == batch_id) & \
                             (batch.batch[batch.edge_index[1]] == batch_id)
        subgraph_edges = batch.edge_index[:, subgraph_edge_mask]
        
        node_mapping = {old_idx.item(): new_idx for new_idx, old_idx in enumerate(node_indices)}
        remapped_edges = torch.tensor([[node_mapping[e.item()] for e in edge] for edge in subgraph_edges.T])
        
        G = nx.Graph()
        G.add_edges_from(remapped_edges.numpy())
        
        nx.set_node_attributes(G, {v: k for k, v in node_mapping.items()}, "original_id")
        
        node_colors = []
        node_borders = []
        
        for node in G.nodes:
            if attr[batch.batch==batch_id][node] == 1:
                node_colors.append("white")  # Fill color
                node_borders.append("red")  # Border color for attr == 1
            else:
                node_colors.append("lightblue")  # Fill color
                node_borders.append("black")  # Default border color
        
        
        pos = nx.kamada_kawai_layout(G) 
        
        nx.draw(
            G, pos,
            node_color=node_colors,
            edgecolors=node_borders,  # Border colors
            node_size=700,
            with_labels=True,
            ax = axs[i]
        )
        
        # axs[i].set_title(f"Class = {batch.y[batch_id]}")
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

def hoyer_sparsity_loss(weights, lambda_=1.0, epsilon=1e-12):
    """
    Hoyer's sparsity loss to promote sparsity.
    
    Args:
        weights (torch.Tensor): The weights to regularize.
        lambda_ (float): Regularization strength.
        epsilon (float): Small value to prevent division by zero.
    
    Returns:
        torch.Tensor: The Hoyer's sparsity loss.
    """
    l1_norm = torch.sum(torch.abs(weights), -1)
    l2_norm = torch.sqrt(torch.sum(weights**2, -1) + epsilon)
    hoyer = (torch.sqrt(torch.tensor(weights.numel())) - l1_norm / l2_norm) / \
            (torch.sqrt(torch.tensor(weights.numel())) - 1 + epsilon)
    loss = lambda_ * (1 - hoyer)
    return loss.mean()


def train_sparsity_epoch(model_tell, loader, device, optimizer, num_classes, conv_reg=1, fc_reg=1):
    model_tell.train()
    
    total_loss = 0
    total_correct = 0
    
    for data in loader:
        try:
            loss = 0
            if data.x is None:
                data.x = torch.ones((data.num_nodes, model_tell.num_features))
            if data.y.numel() == 0: continue
            if data.x.isnan().any(): continue
            if data.y.isnan().any(): continue
            y = data.y.reshape(-1).to(device).long()
            optimizer.zero_grad()

            model_tell.fc.phi_in.tau = 10
            if data.edge_attr is None:
                out = model_tell(data.x.float().to(device), data.edge_index.to(device), None, data.batch.to(device))
            else:
                out = model_tell(data.x.float().to(device), data.edge_index.to(device), data.edge_attr.to(device), data.batch.to(device))       
            pred = out.argmax(-1)
            loss += F.binary_cross_entropy(out.reshape(-1), torch.nn.functional.one_hot(y, num_classes=num_classes).float().reshape(-1)) + F.nll_loss(F.log_softmax(out, dim=-1), y.long())
            for conv in model_tell.convs:
                loss += conv_reg*(torch.sqrt(torch.clamp(conv.nn_0.weight, min=1e-5)).sum(-1).mean())
                loss += conv_reg*(torch.sqrt(torch.clamp(conv.nn_1.weight, min=1e-5)).sum(-1).mean()+ conv.nn_1.phi_in.entropy)
                #loss += (hoyer_sparsity_loss(torch.clamp(conv.nn_0.weight, min=1e-5)) + conv.nn_0.reg_loss) + conv_reg*(torch.sqrt(torch.clamp(conv.nn_0.weight, min=1e-5)).sum(-1).mean())
                #loss += (hoyer_sparsity_loss(torch.clamp(conv.nn_1.weight, min=1e-5)) + conv.nn_1.reg_loss + conv.nn_1.phi_in.entropy) + conv_reg*(torch.sqrt(torch.clamp(conv.nn_1.weight, min=1e-5)).sum(-1).mean()+ conv.nn_1.phi_in.entropy)
            
            loss += (hoyer_sparsity_loss(torch.clamp(model_tell.fc.weight, min=1e-5)) + model_tell.fc.reg_loss + model_tell.fc.phi_in.entropy) + fc_reg*(torch.sqrt(torch.clamp(model_tell.fc.weight, min=1e-5)).sum(-1).mean() + model_tell.fc.phi_in.entropy)

            loss.backward()
            zero_nan_gradients(model_tell)
            optimizer.step()
            total_loss += loss.item() * data.num_graphs / len(loader.dataset)
            total_correct += pred.eq(y).sum().item() / len(loader.dataset)
        except Exception as e:
            print(e)
            pass

    return total_loss, total_correct

@torch.no_grad()
def forward_with_activations(self, x, edge_index, edge_attr, batch, *args, **kwargs):
    returns = []
    xs = []

    if edge_attr is not None and self.phi_edge is not None:
        before_phi_edge= edge_attr
        edge_attr = self.phi_edge(edge_attr)
    if edge_attr is not None and self.negative_concatenate != 0:
        edge_attr = torch.hstack([edge_attr, 1-edge_attr])

    for i, conv in enumerate(self.convs):
        ret = {}
        if i == 0 and not self.input_binary and self.phi_node is not None:
            ret['before_phi_node'] = x
            x = self.phi_node(x)
            ret['after_phi_node'] = x
            ret['after_phi_node_bin'] = x>=0.5
        
        if edge_attr is not None:
            if i ==0 or self.edge_again:
                if self.phi_edge is not None:
                    ret['edge_before_phi'] = before_phi_edge
                ret['edge_attr'] = edge_attr
                ret['edge_attr_bin'] = edge_attr >=0.5

        if self.negative_concatenate == 2 or (self.negative_concatenate == 1 and i==0):
            ret['x'] = torch.hstack([x, 1-x])
        else:
            ret['x'] = x

        edge_index_src, edge_index_dst = edge_index[0], edge_index[1]
        x_i = ret['x'][edge_index_dst]  # destination nodes
        x_j = ret['x'][edge_index_src]  # source nodes

        if i == 0 and edge_attr is not None:
            ret['message_input'] = torch.cat([x_i, x_j, edge_attr], dim=1)
        elif i != 0 and edge_attr is not None and self.edge_again:
            ret['message_input'] = torch.cat([x_i, x_j, edge_attr], dim=1)
        else:
            ret['message_input'] = torch.cat([x_i, x_j], dim=1)

        # print("Message input shape:", ret['message_input'].shape)
        # print("X shape:", ret['x'].shape)
        # print("X_i shape:", x_i.shape)
        # print("X_j shape:", x_j.shape)
        # print("Edge index shape:", edge_index.shape)

        ret['message_input_bin'] = ret['message_input']>=0.5
        message_output = conv.nn_0(ret['message_input'])
        if self.negative_concatenate == 2:
            ret['message_output'] = torch.hstack([message_output, 1-message_output])
        else:
            ret['message_output'] = message_output
        ret['message_output_bin'] = ret['message_output']>=0.5

        ret['x_sum'] = torch_scatter.scatter_add(ret['message_output'], edge_index_dst, 
                                                dim=0, dim_size=ret['x'].size(0))
        #print("x_sum shape:", ret['x_sum'].shape)
        ret['x_bin'] = conv.nn_1.phi_in(ret['x_sum']) >= 0.5

        if i == 0 and edge_attr is not None:
            x, _ = conv(ret['x'], edge_index, edge_attr)
        elif i!=0 and edge_attr is not None and self.edge_again:
            x, _ = conv(ret['x'], edge_index, edge_attr)
        else:
            x, _ = conv(ret['x'], edge_index, None)

        xs.append(x)
        ret['y'] = x
        ret['y_bin'] = x>=0.5
        ret['batch'] = batch
        returns.append(ret)
    
    # Final readout layer
    ret = {}
    x_mean = global_mean_pool(torch.hstack(xs), batch)
    x_max = global_max_pool(torch.hstack(xs), batch)
    x_sum = global_add_pool(torch.hstack(xs), batch)
    x = torch.hstack([x_mean, x_max, x_sum])
    if self.negative_concatenate == 2:
        ret['x'] = torch.hstack([x, 1-x])
    else:
        ret['x'] = x
    ret['x_bin'] = self.fc.phi_in(ret['x']) >= 0.5
    x = self.fc(ret['x'])
    ret['y'] = x
    ret['y_bin'] = x>=0.5
    returns.append(ret)
    return x, returns

def sigmoid(x, tau=10):
    return 1/(1+torch.exp(-tau*x))
    
def forward_tell(self, tau):
    
    def fw(x):
        if self.phi_in is not None:
            x = self.phi_in(x)

        reg_loss = 0
        entropy_loss = 0
        reg_loss += torch.clamp(self.weight, min=1e-5).sum(-1).mean()
        
        if self.phi_in is not None and self.phi_in.entropy is not None:
            entropy_loss += self.phi_in.entropy
            reg_loss += self.phi_in.reg_loss

        self.reg_loss = reg_loss
        w = self.weight
        o = sigmoid(x @ w.t() + self.b, tau=tau)
        
        self.entropy_loss = entropy_loss + -(o*torch.log(o+1e-8) + (1-o)*torch.log(1-o + 1e-8)).mean()
        return o
    
    return fw

def clone_model(model):
    cloned = copy.deepcopy(model)
    cloned.load_state_dict(model.state_dict())
    return cloned

# def clone_model(model):
#     buffer = io.BytesIO()
#     torch.save(model, buffer)
#     buffer.seek(0)
#     cloned_model = torch.load(buffer)
#     return cloned_model

def get_subgraph(data, node_mask):
    nodes_to_keep = torch.where(node_mask)[0]
    
    # Assicurati che node_mask abbia la dimensione corretta
    if node_mask.dim() == 0:
        node_mask = node_mask.unsqueeze(0)
    
    # Usa solo i nodi che esistono effettivamente
    nodes_to_keep = nodes_to_keep[nodes_to_keep < data.x.shape[0]]
    
    if nodes_to_keep.numel() == 0:
        # Nessun nodo da tenere, restituisci grafo vuoto
        return nx.Graph()
    
    new_edge_index, _ = subgraph(nodes_to_keep, data.edge_index, relabel_nodes=True, num_nodes=data.x.shape[0])
    new_x = data.x[node_mask[:data.x.shape[0]]]
    G = to_networkx(Data(x=new_x, edge_index=new_edge_index))
    return G

def find_step_intervals(w, b, xmin, xmax, tau=5, resolution=1000):
    if xmin == xmax:
        y = step(w * xmin + b, tau)
        return [(xmin, xmax)] if y > 0.5 else []

    xs = torch.linspace(xmin, xmax, resolution)
    wxb = w * xs + b
    ys = step(wxb, tau)

    intervals = []
    above = ys[0] > 0.5
    start = xs[0].item() if above else None

    for i in range(1, len(xs)):
        curr = ys[i] > 0.5
        if curr and not above:
            start = xs[i - 1].item()
        elif not curr and above:
            end = xs[i].item()
            intervals.append((start, end))
            start = None
        above = curr

    if above and start is not None:
        intervals.append((start, xs[-1].item()))

    return intervals


def find_minimal_sets(list_of_sets): #Chiaro. 
    minimal_sets = []
    for i, s in enumerate(list_of_sets):
        if not any((set(other)<set(s)) or (s == other and i != j) for j, other in enumerate(list_of_sets)):
            minimal_sets.append(s)
    return minimal_sets

def plot_message_activations(batch_ids, batch, edge_index, attr, save_path=None):
    """
    Visualizza i grafi con archi sottolineati in base alle attivazioni dei messaggi.
    
    Args:
        batch_ids: lista di ID dei batch da visualizzare
        batch: batch di grafi
        edge_index: indici degli archi [2, num_edges]
        attr: activations degli archi (messaggi) da visualizzare (booleani)
    """
    if type(batch_ids) != list:
        batch_ids = [batch_ids]
    num_ids = len(batch_ids)
    cols = 5
    rows = math.ceil(num_ids / cols)
    
    fig, axs = plt.subplots(rows, cols, figsize=(16*5, 8 * rows))
    
    if type(axs) != np.ndarray: 
        axs = np.array([axs])
    axs = axs.flatten()
    for ax in axs:
        ax.set_axis_off()
    
    for i, batch_id in enumerate(batch_ids):
        node_mask = batch.batch == batch_id
        if node_mask.float().sum() == 0: 
            continue
        node_indices = torch.nonzero(node_mask, as_tuple=True)[0]
        
        # Filtra gli archi del subgraph
        subgraph_edge_mask = (batch.batch[edge_index[0]] == batch_id) & \
                             (batch.batch[edge_index[1]] == batch_id)
        subgraph_edges = edge_index[:, subgraph_edge_mask]
        edge_attr_filtered = attr[subgraph_edge_mask]
        
        node_mapping = {old_idx.item(): new_idx for new_idx, old_idx in enumerate(node_indices)}
        
        # Usa DiGraph per mantenere entrambe le direzioni
        G = nx.DiGraph()
        
        # Aggiungi gli archi con le loro attivazioni
        active_edges = []
        inactive_edges = []
        
        for edge_idx in range(subgraph_edges.shape[1]):
            src = subgraph_edges[0, edge_idx].item()
            dst = subgraph_edges[1, edge_idx].item()
            
            src_mapped = node_mapping[src]
            dst_mapped = node_mapping[dst]
            
            G.add_edge(src_mapped, dst_mapped)
            
            # Controlla se l'arco è attivo
            if edge_attr_filtered[edge_idx]:
                active_edges.append((src_mapped, dst_mapped))
            else:
                inactive_edges.append((src_mapped, dst_mapped))
        
        nx.set_node_attributes(G, {v: k for k, v in node_mapping.items()}, "original_id")
        
        node_colors = ["lightblue" for _ in G.nodes]
        node_borders = ["black" for _ in G.nodes]
        
        pos = nx.kamada_kawai_layout(G)
        
        # Disegna nodi
        nx.draw_networkx_nodes(
            G, pos,
            node_color=node_colors,
            edgecolors=node_borders,
            node_size=700,
            ax=axs[i]
        )
        
        nx.draw_networkx_labels(G, pos, ax=axs[i])
        
        # Disegna archi inattivi
        if inactive_edges:
            nx.draw_networkx_edges(
                G, pos,
                edgelist=inactive_edges,
                width=1.0,
                alpha=0.3,
                edge_color='gray',
                ax=axs[i],
                arrows=True,
                arrowsize=10
            )
        
        # Disegna archi attivi
        if active_edges:
            nx.draw_networkx_edges(
                G, pos,
                edgelist=active_edges,
                width=4.0,
                alpha=0.8,
                edge_color='red',
                ax=axs[i],
                arrows=True,
                arrowsize=15
            )
        
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    plt.show()

def print_gnn_rules(all_rules, instance_to_show, class_to_explain=1, save_dir=None):
    """
    Stampa e visualizza le regole logiche estratte dal modello GNN
    
    Args:
        all_rules: dizionario con struttura {
            'fc'
            'conv'
            'phi_node'
            'phi_edge'
        }
        instance_to_show: singolo grafo (torch_geometric.data.Data)
        class_to_explain: classe da spiegare (default=1)
    """
    
    print("\n" + "="*80)
    print(f"LOGICAL RULES FOR CLASS {class_to_explain}")
    print("="*80)
    
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        with open(f'{save_dir}/formulas.txt', 'w') as f:
            f.write(f"LOGICAL RULES FOR CLASS {class_to_explain}\n")
            f.write("="*80 + '\n')
    
    fc_rules = all_rules['fc'].get(class_to_explain, [])
    conj_strs = []
    for rule in fc_rules:
        disj_strs = []
        for literal in rule:
            feat_idx, intervals, mapped = literal['fc_feat_idx'], literal['intervals'], literal['mapped']
            interval_str = " ∪ ".join(
                [f"[{l:.3f}, {u:.3f}]" for l, u in intervals]
            )
            desc = (                
                f"{'+' if mapped[0] == 'pos' else '-'} Feature {feat_idx} "
                f"({mapped[1]}, ConvLayer {mapped[2]} Output {mapped[3]}), "
                f"Activation in {interval_str}")    
            disj_strs.append(desc)
        conj_strs.append('('+" AND ".join(disj_strs)+')')
    rule_description = " OR ".join(conj_strs)
    print(f"\nLogical Rule for Class {class_to_explain}:\n{rule_description}")

    if save_dir:
        with open(f'{save_dir}/formulas.txt', 'a') as f:
            f.write(rule_description + '\n')


    print("\n=== Convolutional Layer Rules/Explanations ===")

    if save_dir:
        with open(f'{save_dir}/formulas.txt', 'a') as f:
            f.write("\n=== Convolutional Layer Rules/Explanations ===\n")

    conv_rules = all_rules['conv']
    for key in conv_rules:
        layer, feat, sublayer = key
        rule_conjs = conv_rules[key]['rules']

        print(f"\n-- ConvLayer {layer}, Output {feat}, Sublayer {sublayer} --")

        if save_dir:
            with open(f'{save_dir}/formulas.txt', 'a') as f:
                f.write(f"\n-- ConvLayer {layer}, Output {feat}, Sublayer {sublayer} --\n")

        print(" OR(")

        if save_dir:
            with open(f'{save_dir}/formulas.txt', 'a') as f:
                f.write(" OR(" + '\n')

        for conj in rule_conjs:
            disj_strs = [] 
            for literal in conj:
                if sublayer == 'nn_1':
                    x_sum_idx, intervals = literal['x_sum_idx'], literal['intervals']
                    interval_str = " ∪ ".join(
                        [f"[{l:.3f}, {u:.3f}]" for l, u in intervals]
                    )
                    desc = f"ConvLayer {layer} Output {x_sum_idx} Sublayer nn_0 sum activation in {interval_str}"
                
                elif sublayer == 'nn_0':
                    msg_input_idx, component, local_idx = literal['msg_input_idx'], literal['component'], literal['local_idx']
                    if layer != 0:
                        desc = (f"ConvLayer {layer-1} Output {local_idx} Component {component}")
                    else:
                        desc = (f"Input {local_idx} Component {component}")

                disj_strs.append('('+desc+')')
            print("  Rule:", '(' + " AND ".join(disj_strs) + '),')
            
            if save_dir:
                with open(f'{save_dir}/formulas.txt', 'a') as f:
                    f.write("  Rule:" + '(' + " AND ".join(disj_strs) + '),' + '\n')

        print("  )")

        if save_dir:
            with open(f'{save_dir}/formulas.txt', 'a') as f:
                f.write("  )" + '\n')

        print("Plotting message activations for above rule...")
        if sublayer == 'nn_1':
            print(conv_rules[key]['all_nodes_mask'])
            plot_activations([0], Batch.from_data_list([instance_to_show]), conv_rules[key]['all_nodes_mask'], save_path=os.path.join(save_dir, f"conv_layer_{key[0]}_feat_{key[1]}_sublayer_{key[2]}.png") if save_dir else None)
        elif sublayer == 'nn_0':
            print("Active messages mask:", conv_rules[key]['all_masks'])
            print("Edge index:", instance_to_show.edge_index)
            plot_message_activations(
                [0], 
                Batch.from_data_list([instance_to_show]), 
                instance_to_show.edge_index,
                conv_rules[key]['all_masks'],
                save_path=os.path.join(save_dir, f"conv_layer_{key[0]}_feat_{key[1]}_sublayer_{key[2]}.png") if save_dir else None
            )

    print("\n=== Phi Nodes Rules/Explanations ===")

    if save_dir:
        with open(f'{save_dir}/formulas.txt', 'a') as f:
            f.write("\n=== Phi Nodes Rules/Explanations ===\n")

    phi_node_rules = all_rules['phi_node']
    for feat_idx in phi_node_rules:
        rule = phi_node_rules[feat_idx]
        base_feat = rule['base_feat']
        intervals = rule['intervals']
        is_negation = rule['is_negation']
        interval_str = " ∪ ".join(
            [f"[{l:.3f}, {u:.3f}]" for l, u in intervals]
        )
        desc = (                
            f"{'NOT ' if is_negation else ''}PhiNode Feature {feat_idx} "
            f"(Base Feature {base_feat}), "
            f"{'NOT ' if is_negation else ''}Activation in {interval_str}")    
        print(f"PhiNode Rule for Feature {desc}")

        if save_dir:
            with open(f'{save_dir}/formulas.txt', 'a') as f:
                f.write(f"PhiNode Rule for Feature {desc}\n")
        print("Plotting phi node activations for above rule...")
        print(rule['all_nodes_mask'])
        plot_activations([0], Batch.from_data_list([instance_to_show]), rule['all_nodes_mask'], save_path=os.path.join(save_dir, f"phi_node_feature_{feat_idx}_base_feat_{base_feat}.png") if save_dir else None)

    
    print("\n=== Phi Edge Rules/Explanations ===")
    
    if save_dir:
        with open(f'{save_dir}/formulas.txt', 'a') as f:
            f.write("\n=== Phi Edge Rules/Explanations ===\n")

    phi_edge_rules = all_rules['phi_edge']
    for feat_idx in phi_edge_rules:
        rule = phi_edge_rules[feat_idx]
        base_feat = rule['base_feat']
        intervals = rule['intervals']
        is_negation = rule['is_negation']
        interval_str = " ∪ ".join(
            [f"[{l:.3f}, {u:.3f}]" for l, u in intervals]
        )
        desc = (                
            f"{'NOT ' if is_negation else ''}PhiEdge Feature {feat_idx} "
            f"(Base Feature {base_feat}), "
            f"{'NOT ' if is_negation else ''}Activation in {interval_str}")    
        print(f"PhiEdge Rule for Feature {desc}")

        if save_dir:
            with open(f'{save_dir}/formulas.txt', 'a') as f:
                f.write(f"PhiEdge Rule for Feature {desc}\n")
        print("Plotting phi node activations for above rule...")
        print("Active edges mask:", rule['all_edges_mask'])
        print("Edge index:", instance_to_show.edge_index)
        plot_message_activations([0], Batch.from_data_list([instance_to_show]), instance_to_show.edge_index, rule['all_edges_mask'], save_path=os.path.join(save_dir, f"phi_edge_feature_{feat_idx}_base_feat_{base_feat}.png") if save_dir else None)

# def forward_tell(self, x, discrete_output=False, tau=10):
#         # x = self.phi_in(torch.hstack([x, 1-x]))
#         if self.phi_in is not None:
#             x = self.phi_in(x)
#         #self.max_in, _ = x.max(0)
#         reg_loss = 0
#         entropy_loss = 0

#         reg_loss += torch.clamp(self.weight, min=1e-5).sum(-1).mean()
        
#         if self.phi_in is not None and self.phi_in.entropy is not None:
#             entropy_loss += self.phi_in.entropy
#             reg_loss += self.phi_in.reg_loss
#         # print('b', reg_loss, entropy_loss)
#         self.reg_loss = reg_loss
#         w = self.weight
#         o = torch.sigmoid(tau*(x @ w.t() + self.b))
        
#         self.entropy_loss = entropy_loss + -(o*torch.log(o+1e-8) + (1-o)*torch.log(1-o + 1e-8)).mean()
#         return o

# def make_lambda(nn, tau):
#     def _forward(x, discrete_output=False):
#         in_features = nn.weight.shape[1]
#         if x.shape[1] < in_features:
#             pad = in_features - x.shape[1]
#             x = torch.cat([x, x.new_zeros(x.size(0), pad)], dim=1)
#         elif x.shape[1] > in_features:
#             x = x[:, :in_features]
#         return forward_tell(nn, x, discrete_output, tau=tau)
#     return _forward

# Initialization

In [ ]:
dataset_name = 'NCI-1'
seed = 0
results_path = os.path.join(get_best_path(dataset_name), str(seed))
data = pickle.load(open(os.path.join(results_path, 'data.pkl'), 'rb'))
args = json.load(open(os.path.join(results_path, 'args.json'), 'r'))
print("Args: ", args)
device = torch.device('cuda') if torch.cuda.is_available else torch.device('cpu')
print("Device: ", device)
model_tell = torch.load(os.path.join(results_path, 'best.pt'), map_location=device)
dataset = get_dataset(dataset_name)
num_classes = dataset.num_classes
num_features = dataset.num_features
num_layers = 5 #Cambiare ogni volta che si cambia modello
hidden_dim = 64 #Cambiare ogni volta che si cambia modello
print("Dataset:", dataset, "Num classes:", num_classes, "Num features:", num_features, "Num layers:", num_layers, "Hidden dim:", hidden_dim)

os.makedirs(f'explanations/{dataset_name}/{seed}', exist_ok=True)
for i in range(num_classes):
    os.makedirs(f'explanations/{dataset_name}/{seed}/classe_{i}', exist_ok=True)

indices = list(range(len(dataset)))
train_indices, val_test_indices = train_test_split(indices, test_size=0.2,
shuffle=True, stratify=dataset.data.y, random_state=1)

val_indices = val_test_indices[:len(val_test_indices)//2]
test_indices = val_test_indices[len(val_test_indices)//2:]

train_dataset = dataset[train_indices]
val_dataset = dataset[val_indices]
test_dataset = dataset[test_indices]
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

save_base_dir = f'explanations/{dataset_name}/{seed}'

   val_acc_mean  test_acc_mean  val_acc_std  test_acc_std  \
0         0.815          0.773     0.018708      0.032711   
1         0.890          0.878     0.023184      0.033279   

                                               fname  
0  results_logic/REDDIT-BINARY/batch_size=16|conv...  
1  results_logic/REDDIT-BINARY/batch_size=16|conv...  
Args:  {'epochs': 5000, 'warmup_epochs': 3000, 'batch_size': 16, 'lr': 0.001, 'l2': 0.0001, 'conv_reg': 0.001, 'fc_reg': 0.01, 'negative_concatenate': 2, 'edge_again': False, 'only_teacher': 2}
Device:  cuda


Extracting data/REDDIT-BINARY/REDDIT-BINARY/REDDIT-BINARY.zip
Processing...


Dataset: REDDIT-BINARY(2000) Num classes: 2 Num features: 0 Num layers: 5 Hidden dim: 64


Done!


In [47]:
# Calcola min e max per ogni feature nel validation set
print("\n" + "="*80)
print("FEATURE STATISTICS FOR VALIDATION SET")
print("="*80)

# Raccogli tutte le feature
all_node_features = []
all_edge_features = []

for data in val_dataset:
    if data.x is not None and data.x.shape[0] > 0:
        all_node_features.append(data.x)
    if hasattr(data, 'edge_attr') and data.edge_attr is not None and data.edge_attr.shape[0] > 0:
        all_edge_features.append(data.edge_attr)

# Concatena tutto
if len(all_node_features) > 0:
    all_node_features = torch.cat(all_node_features, dim=0)
    
    print("\nNODE FEATURES:")
    print(f"Shape: {all_node_features.shape}")
    print(f"Number of features: {all_node_features.shape[1]}")
    
    # Calcola min e max per ogni feature
    node_min = all_node_features.min(dim=0)[0]
    node_max = all_node_features.max(dim=0)[0]
    
    print("\nPer feature:")
    for feat_idx in range(all_node_features.shape[1]):
        print(f"  Feature {feat_idx}: min={node_min[feat_idx]:.4f}, max={node_max[feat_idx]:.4f}")
else:
    print("No node features found")

if len(all_edge_features) > 0:
    all_edge_features = torch.cat(all_edge_features, dim=0)
    
    print("\n" + "-"*80)
    print("EDGE FEATURES:")
    print(f"Shape: {all_edge_features.shape}")
    print(f"Number of features: {all_edge_features.shape[1]}")
    
    # Calcola min e max per ogni feature
    edge_min = all_edge_features.min(dim=0)[0]
    edge_max = all_edge_features.max(dim=0)[0]
    
    print("\nPer feature:")
    for feat_idx in range(all_edge_features.shape[1]):
        print(f"  Feature {feat_idx}: min={edge_min[feat_idx]:.4f}, max={edge_max[feat_idx]:.4f}")
else:
    print("\nNo edge features found")

print("\n" + "="*80)


FEATURE STATISTICS FOR VALIDATION SET

NODE FEATURES:
Shape: torch.Size([4293, 4])
Number of features: 4

Per feature:
  Feature 0: min=1.0000, max=33.0000
  Feature 1: min=0.0000, max=1.0000
  Feature 2: min=0.0000, max=1.0000
  Feature 3: min=0.0000, max=1.0000

No edge features found



# Apply Hoyer Loss

In [6]:
# Salva parametri PRIMA della Hoyer Loss
os.makedirs(save_base_dir, exist_ok=True)

# Conta parametri FC layer
n_w_fc_before = (model_tell.fc.weight>1e-4).sum().item()

# Conta parametri CONV layers
conv_params_before = {}
for l in range(len(model_tell.convs)):
    n_w_0 = (model_tell.convs[l].nn_0.weight>1e-4).sum().item()
    n_w_1 = (model_tell.convs[l].nn_1.weight>1e-4).sum().item()
    conv_params_before[l] = {'nn_0': n_w_0, 'nn_1': n_w_1}

print(f"FC parameters before Hoyer: {n_w_fc_before}")
for l in conv_params_before:
    print(f"Conv Layer {l} - nn_0: {conv_params_before[l]['nn_0']}, nn_1: {conv_params_before[l]['nn_1']}")

FC parameters before Hoyer: 174
Conv Layer 0 - nn_0: 436, nn_1: 654
Conv Layer 1 - nn_0: 1266, nn_1: 603
Conv Layer 2 - nn_0: 1270, nn_1: 635
Conv Layer 3 - nn_0: 1102, nn_1: 608
Conv Layer 4 - nn_0: 1209, nn_1: 609


## FC Layer


In [ ]:
model_tell = model_tell.to(device)
optimizer = torch.optim.Adam([model_tell.fc.weight_sigma, model_tell.fc.weight_exp, model_tell.fc.phi_in.w_], lr=0.01)

best_weights = clone_model(model_tell)
val_acc = test_epoch(model_tell, val_loader, device)
test_acc = test_epoch(model_tell, test_loader, device)
initial_val_acc = val_acc
initial_test_acc = test_acc
n_w =  (model_tell.fc.weight>1e-4).sum().item()
print("Initial val_acc:", val_acc, "test_acc:", test_acc, "n_w:", n_w)
best_situation = (val_acc, -n_w)
patience = max_patience = 100

for i in range(1000):
    train_loss, train_acc = train_sparsity_epoch(model_tell, train_loader, device, optimizer, num_classes, conv_reg=0.1, fc_reg=0.1)
    val_acc = test_epoch(model_tell, val_loader, device)
    test_acc = test_epoch(model_tell, test_loader, device)
    n_w =  (model_tell.fc.weight>1e-4).sum().item()
    # if (val_acc, -n_w) > best_situation:
    if (val_acc, -n_w) > best_situation or (-n_w > best_situation[1] and val_acc >= 0.95*best_situation[0]):
        best_weights = clone_model(model_tell)
        patience = max_patience
        best_situation = (val_acc, -n_w)
    patience -= 1 
    #if i%10 == 0:
    #   print(i, train_loss, train_acc, val_acc, test_acc, n_w, patience)
    if patience == 0:
        break
model_tell = clone_model(best_weights)
val_acc = test_epoch(model_tell, val_loader, device)
test_acc = test_epoch(model_tell, test_loader, device)
n_w =  (model_tell.fc.weight>1e-4).sum().item()
print("Final val_acc:", val_acc, "test_acc:", test_acc, "n_w:", n_w)

Initial val_acc: 0.8179723502304147 test_acc: 0.8317972350230415 n_w: 9


KeyboardInterrupt: 

## CONV Layer

In [ ]:
model_tell = model_tell.to(device)
best_weights = clone_model(model_tell)
val_acc = test_epoch(model_tell, val_loader, device)
test_acc = test_epoch(model_tell, test_loader, device)
print("Initial val_acc:", val_acc, "test_acc:", test_acc)
for l in reversed(range(len(model_tell.convs))):
    n_w_0 =  (model_tell.convs[l].nn_0.weight>1e-4).sum().item()
    n_w_1 =  (model_tell.convs[l].nn_1.weight>1e-4).sum().item()
    print("Layer", l, "n_w_0:", n_w_0, "n_w_1:", n_w_1)

for l in reversed(range(len(model_tell.convs))):
    print("Applying Hoyer Loss on Layer", l)
    optimizer = torch.optim.Adam([model_tell.convs[l].nn_1.weight_sigma, model_tell.convs[l].nn_1.weight_exp, model_tell.convs[l].nn_1.phi_in.w_], lr=0.005)
    val_acc = test_epoch(model_tell, val_loader, device)
    test_acc = test_epoch(model_tell, test_loader, device)
    n_w =  (model_tell.convs[l].nn_1.weight>1e-4).sum().item()
    print("Layer", l, "Initial val_acc:", val_acc, "test_acc:", test_acc, "n_w_1:", n_w)
    best_situation = (val_acc, -n_w)
    #print(best_situation)
    patience = max_patience = 50
    
    for i in range(1000):
        train_loss, train_acc = train_sparsity_epoch(model_tell, train_loader, device, optimizer, num_classes, conv_reg=0.1, fc_reg=0.01)
        val_acc = test_epoch(model_tell, val_loader, device)
        test_acc = test_epoch(model_tell, test_loader, device)
        n_w =  (model_tell.convs[l].nn_1.weight>1e-4).sum().item()
        if (val_acc, -n_w) > best_situation or (-n_w > best_situation[1] and val_acc >= 0.95*best_situation[0]):
            #print((val_acc, -n_w), 'is better than', best_situation)
            best_weights = clone_model(model_tell)
            patience = max_patience
            best_situation = (val_acc, -n_w)
        patience -= 1 
        # if i%10 == 0:
        #     print(i, train_loss, train_acc, val_acc, test_acc, n_w, patience, best_situation, test_epoch(best_weights, val_loader, device))
        if patience == 0:
            break
    
    model_tell = clone_model(best_weights)
    val_acc = test_epoch(model_tell, val_loader, device)
    test_acc = test_epoch(model_tell, test_loader, device)
    n_w =  (model_tell.convs[l].nn_1.weight>1e-4).sum().item()
    print("Layer", l, "Final val_acc:", val_acc, "test_acc:", test_acc, "n_w_1:", n_w)

    optimizer = torch.optim.Adam([model_tell.convs[l].nn_0.weight_sigma, model_tell.convs[l].nn_0.weight_exp], lr=0.005)
    val_acc = test_epoch(model_tell, val_loader, device)
    test_acc = test_epoch(model_tell, test_loader, device)
    n_w =  (model_tell.convs[l].nn_0.weight>1e-4).sum().item()
    print("Layer", l, "Initial val_acc:", val_acc, "test_acc:", test_acc, "n_w_0:", n_w)
    best_situation = (val_acc, -n_w)
    patience = max_patience = 50
    
    for i in range(1000):
        train_loss, train_acc = train_sparsity_epoch(model_tell, train_loader, device, optimizer, num_classes, conv_reg=0.1, fc_reg=0.01)
        val_acc = test_epoch(model_tell, val_loader, device)
        test_acc = test_epoch(model_tell, test_loader, device)
        n_w =  (model_tell.convs[l].nn_0.weight>1e-4).sum().item()
        if (val_acc, -n_w) > best_situation or (-n_w > best_situation[1] and val_acc >= 0.95*best_situation[0]):
            #print((val_acc, -n_w), 'is better than', best_situation)
            best_weights = clone_model(model_tell)
            patience = max_patience
            best_situation = (val_acc, -n_w)
        patience -= 1 
        # if i%10 == 0:
        #     print(i, train_loss, train_acc, val_acc, test_acc, n_w, patience, best_situation, test_epoch(best_weights, val_loader, device))
        if patience == 0:
            break
    model_tell = clone_model(best_weights)
    val_acc = test_epoch(model_tell, val_loader, device)
    test_acc = test_epoch(model_tell, test_loader, device)
    n_w =  (model_tell.convs[l].nn_0.weight>1e-4).sum().item()
    print("Layer", l, "Final val_acc:", val_acc, "test_acc:", test_acc, "n_w_0:", n_w)

model_tell = best_weights.to(device)
best_weights = clone_model(model_tell)
val_acc = test_epoch(model_tell, val_loader, device)
test_acc = test_epoch(model_tell, test_loader, device)
final_val_acc = val_acc
final_test_acc = test_acc
print("Final val_acc:", val_acc, "test_acc:", test_acc)
for l in reversed(range(len(model_tell.convs))):
    n_w_0 =  (model_tell.convs[l].nn_0.weight>1e-4).sum().item()
    n_w_1 =  (model_tell.convs[l].nn_1.weight>1e-4).sum().item()
    print("Final Layer", l, "n_w_0:", n_w_0, "n_w_1:", n_w_1)

In [ ]:
# Salva parametri DOPO la Hoyer Loss
n_w_fc_after = (model_tell.fc.weight>1e-4).sum().item()

conv_params_after = {}
for l in range(len(model_tell.convs)):
    n_w_0 = (model_tell.convs[l].nn_0.weight>1e-4).sum().item()
    n_w_1 = (model_tell.convs[l].nn_1.weight>1e-4).sum().item()
    conv_params_after[l] = {'nn_0': n_w_0, 'nn_1': n_w_1}

# Calcola totali
total_before = n_w_fc_before + sum(conv_params_before[l]['nn_0'] + conv_params_before[l]['nn_1'] for l in conv_params_before)
total_after  = n_w_fc_after  + sum(conv_params_after[l]['nn_0'] + conv_params_after[l]['nn_1'] for l in conv_params_after)
total_reduction = total_before - total_after
total_reduction_percent = total_reduction / total_before * 100

# Salva confronto parametri
params_file = os.path.join(save_base_dir, 'parameters_comparison.txt')
with open(params_file, 'w') as f:
    f.write("="*80 + "\n")
    f.write("PARAMETERS COMPARISON - BEFORE AND AFTER HOYER LOSS\n")
    f.write("="*80 + "\n\n")
    
    f.write(f"FC LAYER:\n")
    f.write(f"  Before: {n_w_fc_before} parameters\n")
    f.write(f"  After:  {n_w_fc_after} parameters\n")
    f.write(f"  Reduction: {n_w_fc_before - n_w_fc_after} ({(n_w_fc_before - n_w_fc_after)/n_w_fc_before*100:.2f}%)\n\n")
    
    for l in conv_params_before:
        f.write(f"CONV LAYER {l}:\n")
        f.write(f"  nn_0 - Before: {conv_params_before[l]['nn_0']} parameters\n")
        f.write(f"  nn_0 - After:  {conv_params_after[l]['nn_0']} parameters\n")
        f.write(f"  nn_0 - Reduction: {conv_params_before[l]['nn_0'] - conv_params_after[l]['nn_0']} ({(conv_params_before[l]['nn_0'] - conv_params_after[l]['nn_0'])/conv_params_before[l]['nn_0']*100:.2f}%)\n")
        
        f.write(f"  nn_1 - Before: {conv_params_before[l]['nn_1']} parameters\n")
        f.write(f"  nn_1 - After:  {conv_params_after[l]['nn_1']} parameters\n")
        f.write(f"  nn_1 - Reduction: {conv_params_before[l]['nn_1'] - conv_params_after[l]['nn_1']} ({(conv_params_before[l]['nn_1'] - conv_params_after[l]['nn_1'])/conv_params_before[l]['nn_1']*100:.2f}%)\n\n")
    
    f.write("="*80 + "\n")
    f.write(f"TOTAL PARAMETERS:\n")
    f.write(f"  Before: {total_before}\n")
    f.write(f"  After:  {total_after}\n")
    f.write(f"  Reduction: {total_reduction} ({total_reduction_percent:.2f}%)\n")
    f.write("="*80 + "\n")
    f.write(f"INITIAL ACCURACIES:\n")
    f.write(f"  Validation: {initial_val_acc}\n")
    f.write(f"  Test:       {initial_test_acc}\n")
    f.write("="*80 + "\n")
    f.write(f"FINAL ACCURACIES:\n")
    f.write(f"  Validation: {final_val_acc}\n")
    f.write(f"  Test:       {final_test_acc}\n")
    f.write("="*80 + "\n")
    
print(f"Parameters comparison saved to {params_file}")
print(f"\nFC parameters: {n_w_fc_before} -> {n_w_fc_after} (reduction: {n_w_fc_before - n_w_fc_after})")
print(f"Total parameters: {total_before} -> {total_after} (reduction: {total_reduction})")



torch.save(model_tell, os.path.join(save_base_dir, 'model_tell_final.pt'))

# Activations for Validation Set

In [58]:
model_tell = torch.load(os.path.join(save_base_dir, 'model_tell_final.pt'), map_location=device)
print(model_tell)
for layer in model_tell.convs:
    n_w_0 =  (layer.nn_0.weight>1e-4).sum().item()
    n_w_1 =  (layer.nn_1.weight>1e-4).sum().item()
    print("Layer", i, "n_w_0:", n_w_0, "n_w_1:", n_w_1)
n_w_fc = (model_tell.fc.weight>1e-4).sum().item()
print("FC n_w:", n_w_fc)

GINTELL(
  (convs): ModuleList(
    (0): CustomGraphConv()
    (1): CustomGraphConv()
    (2): CustomGraphConv()
    (3): CustomGraphConv()
    (4): CustomGraphConv()
  )
  (fc): LogicalLayer(
    (phi_in): Phi()
  )
)
Layer 1 n_w_0: 29 n_w_1: 911
Layer 1 n_w_0: 8132 n_w_1: 806
Layer 1 n_w_0: 3802 n_w_1: 2226
Layer 1 n_w_0: 6504 n_w_1: 1631
Layer 1 n_w_0: 169 n_w_1: 103
FC n_w: 86


In [59]:
with torch.no_grad():
    activations = None
    for batch in val_loader:
        #print(batch)
        if batch.edge_attr is None:
            if batch.x is None:
                batch.x = torch.ones((batch.num_nodes, 10))
            _, rets = forward_with_activations(model_tell.to(device), batch.x.to(device).float(), batch.edge_index.to(device), None, batch.batch.to(device))
        else:
            _, rets = forward_with_activations(model_tell.to(device), batch.x.to(device).float(), batch.edge_index.to(device), batch.edge_attr.to(device), batch.batch.to(device))
        
        if activations is None:
            print(1)
            activations = rets
            for el in activations:
                for l in el:
                    print(l, el[l].shape)
        else:
            print(2) #Al momento non si entra qui- Capire cosa succede
            for l in range(len(rets)):
                print("Rets l", rets[l])
                for k in rets[l]:
                    print("Rets l k", k)
                    if k == 'batch':
                        rets[l]['batch'] += torch.max(activations[l]['batch'])+1
                        activations[l][k] = torch.cat([activations[l][k], rets[l][k]])
                    else:
                        activations[l][k] = torch.vstack([activations[l][k], rets[l][k]])

print("Activations collected.")
print(activations)

1
x torch.Size([23493, 20])
message_input torch.Size([54372, 40])
message_input_bin torch.Size([54372, 40])
message_output torch.Size([54372, 128])
message_output_bin torch.Size([54372, 128])
x_sum torch.Size([23493, 128])
x_bin torch.Size([23493, 128])
y torch.Size([23493, 64])
y_bin torch.Size([23493, 64])
batch torch.Size([23493])
x torch.Size([23493, 128])
message_input torch.Size([54372, 256])
message_input_bin torch.Size([54372, 256])
message_output torch.Size([54372, 128])
message_output_bin torch.Size([54372, 128])
x_sum torch.Size([23493, 128])
x_bin torch.Size([23493, 128])
y torch.Size([23493, 64])
y_bin torch.Size([23493, 64])
batch torch.Size([23493])
x torch.Size([23493, 128])
message_input torch.Size([54372, 256])
message_input_bin torch.Size([54372, 256])
message_output torch.Size([54372, 128])
message_output_bin torch.Size([54372, 128])
x_sum torch.Size([23493, 128])
x_bin torch.Size([23493, 128])
y torch.Size([23493, 64])
y_bin torch.Size([23493, 64])
batch torch.Size

RuntimeError: CUDA out of memory. Tried to allocate 20.00 MiB (GPU 0; 1.95 GiB total capacity; 1.35 GiB already allocated; 19.62 MiB free; 1.49 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

# Extraction Rules

In [ ]:
lista = ['pos', 'neg'] if model_tell.negative_concatenate == 2 else ['pos']
feat_map = []
for pos_neg in lista:
    for readout in ['mean', 'max', 'sum']:
        for l in range(num_layers):
            for d in range(hidden_dim):
                feat_map.append((pos_neg, readout, l, d))
print("Feature map length:", len(feat_map))
print("Feature map:", feat_map)

In [ ]:
class_to_explain = 0
instance_to_show = val_dataset[0].cpu()

try:
    assert instance_to_show.y == class_to_explain
except AssertionError:
    print(f"Warning: instance label {instance_to_show.y} != class_to_explain {class_to_explain}")
    # opzionale: cerca automaticamente un’istanza corretta
    for i, d in enumerate(val_dataset):
        if d.y == class_to_explain:
            instance_to_show = d.cpu()
            print(f"Using instance {i} with label {d.y}")
            break
    else:
        raise ValueError(f"Nessuna istanza trovata con label {class_to_explain}")

batch = Batch.from_data_list([instance_to_show])

if batch.edge_attr is None:
    if batch.x is None:
        batch.x = torch.ones((batch.num_nodes, 10))
    instance_to_show_y, instance_to_show_rets = forward_with_activations(model_tell.to(device), batch.x.to(device), batch.edge_index.to(device), None, batch.batch.to(device))
else:
    instance_to_show_y, instance_to_show_rets = forward_with_activations(model_tell.to(device), batch.x.to(device), batch.edge_index.to(device), batch.edge_attr.to(device), batch.batch.to(device))

In [ ]:
# ============================================================================
# STEP 7: Estrai regole convoluzionali (nn_0 + nn_1) CON UNIFICAZIONE
# Include phi_node e phi_edge per layer 0
# ============================================================================

# Struttura dati per salvare le regole a ogni livello
all_rules = {
    'fc': {},      # Regole del layer finale
    'conv': {},    # Regole convoluzionali: {(layer, feat, 'nn_1'): rules, (layer, feat, 'nn_0'): rules}
    'phi_node': {},  # Regole per binarizzazione input nodi
    'phi_edge': {}   # Regole per binarizzazione edge features
}

# ============================================================================
# Estrai regole FC
# ============================================================================
print("\n" + "="*80)
print("EXTRACTING FC LAYER RULES")
print("="*80)

fc_rules = extract_rules(
    model_tell.fc, 
    feature=class_to_explain,
    activations=activations[-1]['x_bin'].cpu(),
)

print(f"Raw FC rules for class {class_to_explain}: {fc_rules}")
rules_set = set()
    
for rule in fc_rules[0]:
    rule_intervals = []
    
    for feat_idx in rule:
        if not activations[-1]['x_bin'][:, feat_idx].all().item():
            phi_in = model_tell.fc.phi_in
            intervals = find_step_intervals(
                phi_in.w[feat_idx].cpu(), 
                phi_in.b[feat_idx].cpu(), 
                activations[-1]['x'][:, feat_idx].min().cpu(), 
                activations[-1]['x'][:, feat_idx].max().cpu(), 
                tau=10, resolution=1000
            )
            
            rule_intervals.append((
                feat_idx,
                tuple(intervals),
                feat_map[feat_idx]
            ))
    
    if rule_intervals:
        rules_set.add(frozenset(rule_intervals))

rules_with_intervals = []
for rule_frozenset in rules_set:
    rule_dicts = []
    for feat_idx, intervals, mapped in sorted(rule_frozenset, key=lambda x: x[0]):
        rule_dicts.append({
            'fc_feat_idx': feat_idx,
            'intervals': list(intervals),
            'mapped': mapped
        })
    rules_with_intervals.append(rule_dicts)

# Applica find_minimal_sets per rimuovere regole ridondanti
rules_as_sets = [
    frozenset((lit['fc_feat_idx'], tuple(tuple(interval) for interval in lit['intervals']), lit['mapped']) 
            for lit in rule) 
    for rule in rules_with_intervals
]

minimal_rules_sets = find_minimal_sets(rules_as_sets)

# Riconverti in formato dict
minimal_rules_with_intervals = []
for rule_set in minimal_rules_sets:
    rule_dicts = []
    for feat_idx, intervals, mapped in sorted(rule_set, key=lambda x: x[0]):
        rule_dicts.append({
            'fc_feat_idx': feat_idx,
            'intervals': [list(interval) for interval in intervals],
            'mapped': mapped
        })
    minimal_rules_with_intervals.append(rule_dicts)

print(f"FC Class {class_to_explain}: Raw rules: {len(rules_with_intervals)} → Minimal: {len(minimal_rules_with_intervals)}")

all_rules['fc'][class_to_explain] = minimal_rules_with_intervals
print(f"Class {class_to_explain} Rules: {all_rules['fc'][class_to_explain]}")

# ============================================================================
# Estrai regole CONV (nn_1 e nn_0)
# ============================================================================
print("\n" + "="*80)
print("EXTRACTING CONVOLUTIONAL LAYER RULES FOR CLASS ", class_to_explain)
print("="*80)

# Inizializza work queue
work_queue = []
for rule in all_rules['fc'][class_to_explain]:
    for literal in rule:
        pos_neg, readout, layer, feat = literal['mapped']
        work_queue.append((layer, feat, 'nn_1'))

work_queue = list(set(work_queue))
print(f"Initial work queue: {len(work_queue)} items (only for class {class_to_explain})")

processed = set()

while work_queue:
    layer, feat, sublayer = work_queue.pop(0)

    key = (layer, feat, sublayer)
    if key in processed:
        continue
    processed.add(key)

    print(f"\n--- Processing: Layer {layer}, Feat {feat}, Sublayer {sublayer} ---")
    
    # ========================================================================
    # Caso 1: nn_1 (ha phi_in, binarizza x_sum)
    # ========================================================================
    if sublayer == 'nn_1':
        logical_layer = model_tell.convs[layer].nn_1
        phi_in = logical_layer.phi_in
        ho = logical_layer.weight.shape[0]

        rules = extract_rules(
            logical_layer, 
            feature=feat % ho,
        )

        
        print(f"Raw rules: {rules}")

        rules_set = set()
        
        for rule in rules[0]:
            rule_intervals = []
            for x_sum_idx in rule:
                if not activations[layer]['x_bin'][:, x_sum_idx].all().item():
                    #print(activations[layer]['x_bin'][:, x_sum_idx])
                    intervals = find_step_intervals(
                        phi_in.w[x_sum_idx].cpu(), 
                        phi_in.b[x_sum_idx].cpu(), 
                        activations[layer]['x_sum'][:, x_sum_idx].min().cpu(), 
                        activations[layer]['x_sum'][:, x_sum_idx].max().cpu(), 
                        tau=10, resolution=1000
                    )
                    
                    rule_intervals.append((
                        x_sum_idx,
                        tuple(intervals),
                    ))

                    # Aggiungi nn_0 alla coda
                    if (layer, x_sum_idx, 'nn_0') not in processed:
                        work_queue.append((layer, x_sum_idx, 'nn_0'))
            
            if rule_intervals:
                rules_set.add(frozenset(rule_intervals))
        
        rules_with_intervals = []
        for rule_frozenset in rules_set:
            rule_list = []
            for x_sum_idx, intervals in sorted(rule_frozenset, key=lambda x: x[0]):
                rule_list.append({
                    'x_sum_idx': x_sum_idx,
                    'intervals': list(intervals)
                })
            rules_with_intervals.append(rule_list)
        

        if feat < ho:
            all_nodes_mask = instance_to_show_rets[layer]['y_bin'][:, feat]
        else:
            all_nodes_mask = ~instance_to_show_rets[layer]['y_bin'][:, feat % ho]

        print(f"All nodes mask for conv {key}: {all_nodes_mask}")
  
        
        # Applica find_minimal_sets per rimuovere regole ridondanti
        rules_as_sets = [
            frozenset((lit['x_sum_idx'], tuple(tuple(interval) for interval in lit['intervals'])) 
                    for lit in rule) 
            for rule in rules_with_intervals
        ]

        minimal_rules_sets = find_minimal_sets(rules_as_sets)

        # Riconverti in formato dict
        minimal_rules_with_intervals = []
        for rule_set in minimal_rules_sets:
            rule_list = []
            for x_sum_idx, intervals in sorted(rule_set, key=lambda x: x[0]):
                rule_list.append({
                    'x_sum_idx': x_sum_idx,
                    'intervals': [list(interval) for interval in intervals]
                })
            minimal_rules_with_intervals.append(rule_list)

        print(f"Raw rules: {len(rules_with_intervals)} → Minimal: {len(minimal_rules_with_intervals)}")

        all_rules['conv'][key] = {
            'rules': minimal_rules_with_intervals,
            'all_nodes_mask': all_nodes_mask,
            'batch': activations[layer]['batch']
        }
        print(f"Rules for conv {key}: {all_rules['conv'][key]['rules']}")
        # print(f"All nodes mask for conv {key}: {all_rules['conv'][key]['all_nodes_mask']}")
        # print(f"Len mask batch: {len(all_rules['conv'][key]['all_nodes_mask'])}, Len batch: {len(all_rules['conv'][key]['batch'])}")


    # ========================================================================
    # Caso 2: nn_0 (NO phi_in, riceve input binari [x_i, x_j, edge_attr])
    # ========================================================================
    elif sublayer == 'nn_0':
        logical_layer = model_tell.convs[layer].nn_0
        ho = logical_layer.weight.shape[0]

        rules = extract_rules(
            logical_layer, 
            feature=feat % ho,
        )
        
        print(f"Raw rules: {rules}")

        # Calcola dimensioni
        if layer == 0:
            if model_tell.negative_concatenate == 2 or model_tell.negative_concatenate == 1:
                x_dim = 2 * num_features
            else:
                x_dim = num_features
        else:
            if model_tell.negative_concatenate == 2:
                x_dim = 2 * hidden_dim
            else:
                x_dim = hidden_dim
        
        num_features_edge = model_tell.num_features_edge
        edge_dim = (2 * num_features_edge if model_tell.negative_concatenate != 0 
                   else num_features_edge) if num_features_edge > 0 else 0

        rules_set = set()
        
        for rule in rules[0]:
            rule_deps = []
            
            for msg_input_idx in rule:
                if not activations[layer]['message_input_bin'][:, msg_input_idx].all().item():
                    #print(instance_to_show_rets[layer]['message_input_bin'][:, msg_input_idx])
                    # Decodifica posizione
                    if msg_input_idx < x_dim:
                        component = 'x_i'
                        local_idx = msg_input_idx
                    elif msg_input_idx < 2 * x_dim:
                        component = 'x_j'
                        local_idx = msg_input_idx - x_dim
                    else:
                        component = 'edge_attr'
                        local_idx = msg_input_idx - 2 * x_dim
            
                    rule_deps.append((
                        msg_input_idx,
                        component,
                        local_idx
                    ))
                    
                    # ====================================================
                    # GESTIONE PHI_NODE e PHI_EDGE per LAYER 0
                    # ====================================================
                    if layer == 0:
                        # Siamo al primo layer: gli input vengono direttamente dai dati
                        if component in ['x_i', 'x_j']:
                            # Input dei nodi: potrebbe passare per phi_node
                            if not model_tell.input_binary and model_tell.phi_node is not None:
                                # Marca che serve estrarre regole di phi_node
                                work_queue.append(('phi_node', local_idx, None))
                        
                        elif component == 'edge_attr':
                            # Edge features: potrebbero passare per phi_edge
                            if not model_tell.edge_binary and model_tell.phi_edge is not None:
                                # Marca che serve estrarre regole di phi_edge
                                work_queue.append(('phi_edge', local_idx, None))
                    
                    # Risali al layer precedente (se layer > 0)
                    elif component in ['x_i', 'x_j'] and layer > 0:
                        if (layer - 1, local_idx, 'nn_1') not in processed:
                            work_queue.append((layer - 1, local_idx, 'nn_1'))

            if rule_deps:
                rules_set.add(frozenset(rule_deps))
        
        rules_with_dependencies = []
        for rule_frozenset in rules_set:
            rule_list = []
            for msg_input_idx, component, local_idx in sorted(rule_frozenset, key=lambda x: x[0]):
                rule_list.append({
                    'msg_input_idx': msg_input_idx,
                    'component': component,
                    'local_idx': local_idx
                })
            rules_with_dependencies.append(rule_list)
        
        rules_as_sets = [
            set((lit['msg_input_idx'], lit['component'], lit['local_idx']) 
                for lit in rule) 
            for rule in rules_with_dependencies
        ]

        print(f"Rules with dependencies for conv {key}: {rules_as_sets}")
        minimal_rules_sets = find_minimal_sets(rules_as_sets)

        # Riconverti in formato dict
        minimal_rules_dicts = []
        for rule_set in minimal_rules_sets:
            rule_list = []
            for msg_input_idx, component, local_idx in sorted(rule_set):
                rule_list.append({
                    'msg_input_idx': msg_input_idx,
                    'component': component,
                    'local_idx': local_idx
                })
            minimal_rules_dicts.append(rule_list)
    
        print(f"Raw rules: {len(rules_with_dependencies)} → Minimal: {len(minimal_rules_dicts)}")
        
       
        all_messages_mask = instance_to_show_rets[layer]['message_output_bin'][:, feat]

        # Per nn_0 non abbiamo batch assignment diretto per i messaggi
        # Ma possiamo ricavarlo da edge_index se necessario
        # Per ora salviamo solo la maschera
        all_rules['conv'][key] = {
            'rules': minimal_rules_dicts,
            'all_masks': all_messages_mask,  # [num_all_messages]
            'batch': activations[layer]['batch'] if 'batch' in activations[layer] else None
        }
        
        print("all_messages_mask:", all_messages_mask)
        # print(f"All nodes mask for conv {key}: {all_rules['conv'][key]['all_masks']}")
        # print(f"Len Masks for conv {key}: {len(all_rules['conv'][key]['all_masks'])} ")
        # print(f"Len mask batch: {len(all_rules['conv'][key]['batch'])}")
    
    elif layer == 'phi_node':
        print(f"Extracting phi_node rules for feature {feat}")
        phi_node = model_tell.phi_node

        num_features = model_tell.num_features

        if model_tell.negative_concatenate in [1, 2]:
            base_feat = feat % num_features
        else:
            base_feat = feat

        intervals = find_step_intervals(
                phi_node.w[base_feat].cpu(),
                phi_node.b[base_feat].cpu(),
                activations[0]['before_phi_node'][:, base_feat].min().cpu(),
                activations[0]['before_phi_node'][:, base_feat].max().cpu(),
                tau=10,
                resolution=1000
        )

        if feat < num_features:
            all_nodes_mask = instance_to_show_rets[0]['after_phi_node_bin'][:, feat]
        else:
            all_nodes_mask = ~instance_to_show_rets[0]['after_phi_node_bin'][:, feat % num_features]


        all_rules['phi_node'][feat] = {
                'base_feat': base_feat,
                'intervals': intervals,
                'is_negation': feat >= num_features if model_tell.negative_concatenate in [1, 2] else False,
                'all_nodes_mask': all_nodes_mask
        }
            
        print(f"Feature {feat} (base: {base_feat}): {intervals}")

    elif layer == 'phi_edge':
        print(f"Extracting phi_edge rules for feature {feat}")
        phi_edge = model_tell.phi_edge
        num_features_edge = model_tell.num_features_edge

        if model_tell.negative_concatenate != 0:
            base_feat = feat % num_features_edge
        else:
            base_feat = feat

        intervals = find_step_intervals(
                phi_edge.w[base_feat].cpu(),
                phi_edge.b[base_feat].cpu(),
                activations[0]['edge_before_phi'][:, base_feat].min().cpu(),
                activations[0]['edge_before_phi'][:, base_feat].max().cpu(),
                tau=10,
                resolution=1000
        )
        if feat < num_features_edge:
            all_edges_mask = instance_to_show_rets[0]['edge_attr_bin'][:, feat]
        else:
            all_edges_mask = ~instance_to_show_rets[0]['edge_attr_bin'][:, feat % num_features_edge]

        all_rules['phi_edge'][feat] = {
                'base_feat': base_feat,
                'intervals': intervals,
                'is_negation': feat >= num_features_edge if model_tell.negative_concatenate != 0 else False,
                'all_edges_mask': all_edges_mask
        }
        
print("\n" + "="*80)
print(f"EXTRACTION COMPLETE: Processed {len(processed)} layer/feature combinations")
print("="*80)

In [ ]:
save_dir = os.path.join(save_base_dir, f"classe_{class_to_explain}")
save_path = f'{save_dir}/all_rules.pkl'
print(f"Saving all rules to {save_path}")
with open(save_path, 'wb') as f:
    pickle.dump(all_rules, f)
#print_gnn_rules(all_rules, instance_to_show, class_to_explain, save_dir=save_dir)

In [ ]:
print("\n" + "="*80)
print("FINDING REPRESENTATIVE SUBGRAPHS FOR FC RULES")
print("="*80)

model_tell = model_tell.to(device)

# Per ogni classe
fc_rules = all_rules['fc'].get(class_to_explain, [])

if not fc_rules:
    print(f"\nClass {class_to_explain}: No rules found")
    exit(0)

print(f"\n{'='*70}")
print(f"CLASS {class_to_explain}")
print(f"{'='*70}")

# Per ogni regola della classe
for rule_idx, rule in enumerate(fc_rules):
    print(f"\n--- Rule idx {rule_idx + 1} for Class {class_to_explain} ---")
    
    # Estrai feature coinvolte nella regola
    # rule è una lista di dict: [{'fc_feat_idx': ..., 'intervals': ..., 'mapped': ...}, ...]
    ll_feats = [literal['fc_feat_idx'] for literal in rule]
    print(f"Features involved: {ll_feats}")
    
    # Mostra la regola in formato leggibile
    rule_desc = []
    for literal in rule:
        mapped = literal['mapped']
        pos_neg, readout, layer, feat = mapped
        rule_desc.append(f"{pos_neg}_{readout}_L{layer}_F{feat}")
    print(f"Rule: {' AND '.join(rule_desc)}")
    
    representatives = []
    n_matching = 0
    
    # Cerca nel validation set
    for data in val_dataset:

        if data.x is None or data.x.shape[0] == 0:
            continue
        
        if data.y.item() != class_to_explain:
            continue
        
        batch = Batch.from_data_list([data])
        
        try:
            # Forward pass
            if batch.edge_attr is None:
                y, rets = forward_with_activations(
                    model_tell, 
                    batch.x.to(device).float(), 
                    batch.edge_index.to(device), 
                    None, 
                    batch.batch.to(device)
                )
            else:
                y, rets = forward_with_activations(
                    model_tell, 
                    batch.x.to(device).float(), 
                    batch.edge_index.to(device), 
                    batch.edge_attr.to(device).float(), 
                    batch.batch.to(device)
                )
        except Exception as e:
            print(f"  Error processing graph: {e}")
            continue
        
        # Controlla se tutte le feature della regola sono attive a livello FC
        if not rets[-1]['x_bin'][:, ll_feats].all():
            continue
        # Trova i nodi che contribuiscono a questa regola
        node_mask = torch.ones(data.x.shape[0]).bool().to(device)
        
        for literal in rule:
            mapped = literal['mapped']
            pos_neg, readout, layer, feat = mapped
            
            m = rets[layer]['y_bin'][:, feat].detach().cpu()
            
            # Se tutti i nodi sono attivi, non è informativo
            if m.all():
                continue
            
            # Aggiorna maschera (AND logico)
            node_mask = node_mask.cpu() & m
        
        # Se nessun nodo soddisfa la regola, skippa
        if node_mask.sum().item() == 0:
            continue
        
        n_matching += 1
        
        # Estrai sottografo
        G = get_subgraph(data.cpu(), node_mask.cpu())
        
        # Controlla isomorfismo con rappresentanti già trovati
        isomorphic = False
        for G_repr, _, _, _ in representatives:
            if nx.is_isomorphic(G, G_repr):
                isomorphic = True
                break
        
        # Aggiungi solo se non isomorfo
        if not isomorphic:
            representatives.append((G, batch.cpu(), node_mask.cpu(), y.cpu()))
    
    print(f"Total matching graphs: {n_matching}")
    print(f"Unique subgraph structures: {len(representatives)}")
    
    # Mostra i primi 10 rappresentanti
    representatives = representatives[:10]
    
    if len(representatives) > 0:
        print(f"Plotting {len(representatives)} representative subgraphs...")
        
        # Prepara batch per plot
        batches_to_plot = [rep[1] for rep in representatives]
        masks_to_plot = [rep[2] for rep in representatives]
        
        # Crea un unico batch combinato
        combined_batch = Batch.from_data_list([b[0] for b in batches_to_plot])
        combined_mask = torch.cat(masks_to_plot)
        
        # Plot
        plot_activations(
            list(range(len(representatives))), 
            combined_batch, 
            combined_mask,
            save_path=os.path.join(save_dir, f"class_{class_to_explain}_rule_{rule_idx + 1}_fc_representatives.png")
        )
    else:
        print("  No representative subgraphs found")

print("\n" + "="*80)
print("REPRESENTATIVE SUBGRAPH SEARCH COMPLETE")
print("="*80)

In [ ]:
# from rdkit.Chem import rdchem

# # Mapping per i tipi di atomi (node features)
# atom_types = [1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53]
# atom_mapping = {i: atom_type for i, atom_type in enumerate(atom_types)}
# print("Atom mapping:", atom_mapping)

# # Mapping simboli atomici (opzionale, più leggibile)
# atom_symbols = {
#     1: 'H', 5: 'B', 6: 'C', 7: 'N', 8: 'O', 9: 'F',
#     14: 'Si', 15: 'P', 16: 'S', 17: 'Cl', 35: 'Br', 53: 'I'
# }

# # Mapping per i tipi di bond (edge features)
# bond_types = [
#     rdchem.BondType.SINGLE,
#     rdchem.BondType.DOUBLE,
#     rdchem.BondType.TRIPLE,
#     rdchem.BondType.AROMATIC
# ]
# bond_mapping = {i: bond_type for i, bond_type in enumerate(bond_types)}
# bond_names = {
#     0: 'SINGLE',
#     1: 'DOUBLE',
#     2: 'TRIPLE',
#     3: 'AROMATIC'
# }

# # Versione migliorata del tuo codice
# from torch import argmax

# for i in range(instance_to_show['edge_index'].shape[1]):
#     src = instance_to_show['edge_index'][0, i].item()
#     dst = instance_to_show['edge_index'][1, i].item()
#     edge_feat = instance_to_show_rets[0]['edge_attr'][i]
#     edge_idx = argmax(edge_feat).item()
    
#     src_idx = argmax(instance_to_show['x'][src]).item()
#     dst_idx = argmax(instance_to_show['x'][dst]).item()
    
#     print(f"Edge {i}: {src} -> {dst}")
#     print(f"  Bond type: {bond_names[edge_idx]} (idx: {edge_idx})")
#     # print(f"  Source atom: {atom_symbols[atom_mapping[src_idx]]} (atomic num: {atom_mapping[src_idx]})")
#     # print(f"  Dest atom: {atom_symbols[atom_mapping[dst_idx]]} (atomic num: {atom_mapping[dst_idx]})")
#     print("-" * 50)

In [ ]:
# from torch import argmax


# atom_types = [1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53]
# atom_symbols = {
#     1: 'H', 5: 'B', 6: 'C', 7: 'N', 8: 'O', 9: 'F',
#     14: 'Si', 15: 'P', 16: 'S', 17: 'Cl', 35: 'Br', 53: 'I'
# }


# for i in range(instance_to_show['x'].shape[0]):
#     atom_idx = argmax(instance_to_show['x'][i]).item()
#     atomic_num = atom_types[atom_idx]
#     symbol = atom_symbols[atomic_num]
    
#     print(f"Node {i}: {symbol} (atomic number: {atomic_num})")